In [ ]:
# ============================================
# 1. IMPORT LIBRARIES
# ============================================
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ============================================
# 2. LOAD DATASET (IMDB)
# ============================================
vocab_size = 10000
max_len = 250  # Increased slightly for better context

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=vocab_size)

# ============================================
# 3. PREPROCESSING
# ============================================
# Change: Use 'pre' padding. RNNs/LSTMs perform better when the
# relevant data is at the end of the sequence (near the prediction point).
x_train = pad_sequences(x_train, maxlen=max_len, padding='pre')
x_test  = pad_sequences(x_test,  maxlen=max_len, padding='pre')

# ============================================
# 4. BUILD OPTIMIZED MODEL
# ============================================
model = models.Sequential([
    layers.Input(shape=(max_len,)),

    # 1. Embedding Layer
    layers.Embedding(input_dim=vocab_size, output_dim=128),

    # 2. Bidirectional LSTM
    # Bidirectional allows the model to look at the sentence forward and backward.
    # LSTM solves the vanishing gradient problem of SimpleRNN.
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.GlobalMaxPool1D(), # Extract the most important features

    # 3. Dense Layers with Dropout
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),

    # 4. Output Layer
    layers.Dense(1, activation='sigmoid')
])

model.summary()

# ============================================
# 5. COMPILE & CALLBACKS
# ============================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early Stopping prevents overfitting by stopping training when val_loss stops improving
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# ============================================
# 6. TRAIN MODEL
# ============================================
history = model.fit(
    x_train, y_train,
    epochs=10, # 50 is too high; EarlyStopping will handle the exit
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

# ============================================
# 7. EVALUATE MODEL
# ============================================
loss, accuracy = model.evaluate(x_test, y_test)
print(f"\nImproved Test Accuracy: {accuracy * 100:.2f}%")

# ============================================
# 8. PREDICT NEW TEXT (STAYS THE SAME)
# ============================================
word_index = tf.keras.datasets.imdb.get_word_index()

def encode_text(text):
    # The IMDB dataset uses index 2 for unknown words
    words = text.lower().split()
    encoded = [word_index.get(word, 2) + 3 for word in words] # Offset by 3 for IMDB format
    padded = pad_sequences([encoded], maxlen=max_len, padding='pre')
    return padded

def predict_sentiment(text):
    processed = encode_text(text)
    prediction = model.predict(processed, verbose=0)[0][0]
    print(f"\nReview: {text}")
    print(f"Score: {prediction:.4f}")
    print("Sentiment: Positive 😊" if prediction >= 0.5 else "Sentiment: Negative 😞")

predict_sentiment("This movie was excellent and amazing")
predict_sentiment("This movie was very bad and boring")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 250, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 250, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,137 (5.29 MB)

 Trainable params: 1,387,137 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.7706 - loss: 0.4660 - val_accuracy: 0.8666 - val_loss: 0.3115
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.9016 - loss: 0.2579 - val_accuracy: 0.8824 - val_loss: 0.2865
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.9391 - loss: 0.1705 - val_accuracy: 0.8782 - val_loss: 0.3336
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.9614 - loss: 0.1179 - val_accuracy: 0.8740 - val_loss: 0.3543
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9773 - loss: 0.0742 - val_accuracy: 0.8776 - val_loss: 0.4022
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8758 - loss: 0.2955

Improved Test Accuracy: 87.58%
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Review: This movie was excellent and amazing
Score: 0.9405
Sentiment: Positive 😊

Review: This movie was very bad and boring
Score: 0.0240
Sentiment: Negative 😞
